# AgentPulse — Reasoning Strategy Benchmark (Kaggle GPU)

Runs the exact same Direct / CoT / AoT comparison as the local CPU run (`experiments/reasoning_strategies.py`),
using the real project code (strategies, evaluator, risk aggregation) — only the model-loading backend differs
(4-bit `Qwen/Qwen3-8B` on GPU instead of a Q4_K_M GGUF on CPU), so results are directly comparable.

## Setup (do this before Run All)

1. On your machine, go to the GitHub repo page and use **Code > Download ZIP** (this is a private repo, so you must be logged in and be the owner/have access).
2. On Kaggle: **Create > New Dataset**, upload that zip, name it `agentpulse`.
3. In this notebook: **Add Input > Your Datasets > agentpulse**.
4. **Settings (right sidebar) > Accelerator > GPU T4 x2** (or any available GPU).
5. **Run All**.

Output is written to `/kaggle/working/reasoning_strategy_results.json` — download it from the Output panel when done,
and drop it into `experiments/results/reasoning_strategy_results.json` locally to regenerate the report.

In [ ]:
!pip install -q bitsandbytes accelerate sentencepiece

In [ ]:
import os, glob

# Locate the project root under /kaggle/input, regardless of how the zip was named/extracted.
project_root = None
for backend_dir in glob.glob('/kaggle/input/**/backend', recursive=True):
    root = os.path.dirname(backend_dir)
    if os.path.isdir(os.path.join(root, 'sdk')) and os.path.isdir(os.path.join(root, 'reasoning')):
        project_root = root
        break

assert project_root, (
    "Could not find the project under /kaggle/input. "
    "Make sure you added the 'agentpulse' dataset (Add Input) and that the zip contains "
    "the backend/, sdk/, reasoning/, datasets/ folders."
)
print("Project root:", project_root)

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{project_root}/sdk"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{project_root}/backend"], check=True)

sys.path.insert(0, project_root)
sys.path.insert(0, f"{project_root}/backend")
sys.path.insert(0, f"{project_root}/sdk/src")
print("Installed and added to path.")

In [ ]:
import json, time, statistics
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from llm_adapters.base import LLMAdapter, GenerationResult
from reasoning import get_reasoning_strategy
from app.services.evaluator import EvaluationPipeline
from app.services.drift import DriftDetector
from app.services.alerting import AlertEngine
from app.services.grounding import load_models

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## GPU adapter

Implements the same `LLMAdapter` interface the CPU (`LocalGGUFAdapter`) and HF (`LocalHFAdapter`) adapters use,
so `reasoning/*.py` and the evaluator work identically regardless of which one is plugged in. Uses 4-bit
quantization (`bitsandbytes`) since a T4 has 16GB VRAM — full bf16 for an 8B model is too tight for safe headroom.

In [ ]:
class KaggleGPUAdapter(LLMAdapter):
    """Real Qwen3-8B inference on a Kaggle GPU, 4-bit quantized via bitsandbytes."""

    def __init__(self, model_id: str = "Qwen/Qwen3-8B", **kwargs):
        super().__init__(model_id=model_id, device="cuda", quantization="4bit", **kwargs)
        bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
        print(f"Loading {model_id} in 4-bit on GPU...")
        t0 = time.perf_counter()
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id, quantization_config=bnb_config, device_map="cuda",
        )
        self.load_time_s = time.perf_counter() - t0
        print(f"Loaded in {self.load_time_s:.1f}s")

    def generate(self, prompt, **kwargs):
        return self.generate_with_metadata(prompt, **kwargs).text

    def generate_with_metadata(self, prompt, prompt_version="v1.0", dataset_version=None, **kwargs):
        max_tokens = kwargs.get("max_tokens", self.default_max_tokens)
        temperature = kwargs.get("temperature", self.default_temperature)
        top_p = kwargs.get("top_p", self.default_top_p)

        # Qwen3 ships with "thinking mode" on by default; disable it the same way
        # the local GGUF adapter does, so outputs are directly comparable.
        messages = [{"role": "user", "content": f"{prompt} /no_think"}]
        text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(text, return_tensors="pt").to("cuda")

        t0 = time.perf_counter()
        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                temperature=max(0.01, temperature),
                top_p=top_p,
                do_sample=temperature > 0,
                pad_token_id=self.tokenizer.eos_token_id,
            )
        latency_ms = (time.perf_counter() - t0) * 1000.0

        gen_ids = output_ids[0][inputs["input_ids"].shape[1]:]
        output_text = self.tokenizer.decode(gen_ids, skip_special_tokens=True)
        if "</think>" in output_text:
            output_text = output_text.split("</think>", 1)[1].strip()
        else:
            output_text = output_text.strip()

        tokens_in = int(inputs["input_ids"].shape[1])
        tokens_out = int(gen_ids.shape[0])
        tokens_per_sec = (tokens_out / (latency_ms / 1000.0)) if latency_ms > 0 and tokens_out else 0.0

        return GenerationResult(
            text=output_text,
            model_id=self.model_id,
            provider="kaggle_gpu",
            runtime="transformers+bitsandbytes",
            device="cuda",
            quantization="4bit",
            temperature=temperature,
            top_p=top_p,
            max_tokens=max_tokens,
            prompt_version=prompt_version,
            dataset_version=dataset_version,
            latency_ms=latency_ms,
            tokens_in=tokens_in,
            tokens_out=tokens_out,
            raw_metadata={"tokens_per_sec": round(tokens_per_sec, 2)},
        )


adapter = KaggleGPUAdapter()

In [ ]:
load_models(use_onnx=False, sync=True)

with open(f"{project_root}/datasets/v1.0_test.json") as f:
    dataset = json.load(f)
cases = dataset["cases"]
print(f"{len(cases)} test cases loaded")

drift_detector = DriftDetector(window_size=20, min_samples_for_alert=5)
alert_engine = AlertEngine(cooldown_seconds=0)
pipeline = EvaluationPipeline(drift_detector, alert_engine)

warm = adapter.generate_with_metadata(prompt="Reply with the single word: ready.", max_tokens=8)
print(f"Warm-up: {warm.latency_ms:.0f}ms, {warm.tokens_out} tokens -> {warm.raw_metadata['tokens_per_sec']} tok/s")

## Benchmark loop

Same structure as `experiments/reasoning_strategies.py` — identical strategy calls, identical evaluator, identical
per-case/per-run statistics. `N_RUNS` and `MAX_TOKENS` match the local run so the two are directly comparable.

In [ ]:
N_RUNS = 5
MAX_TOKENS = 200
strategies = ["direct", "cot", "aot"]
results_by_strategy = {s: [] for s in strategies}

def _stdev(xs):
    return round(float(statistics.stdev(xs)), 2) if len(xs) > 1 else 0.0

overall_start = time.perf_counter()

for strat_name in strategies:
    strat = get_reasoning_strategy(strat_name)
    print(f"\n=== {strat_name.upper()} ===")

    for case in cases:
        run_latencies, run_tin, run_tout, run_risks, run_contra = [], [], [], [], []

        for run_idx in range(N_RUNS):
            output = strat.execute(
                adapter=adapter,
                task_prompt=case["input_query"],
                context=case.get("evidence"),
                max_tokens=MAX_TOKENS,
            )
            eval_res = pipeline.evaluate_span(
                span_id=f"{strat_name}_{case['id']}_{run_idx}",
                trace_id=f"trace_{strat_name}_{run_idx}",
                agent_id="eval_agent",
                input_text=case.get("evidence") or case["input_query"],
                output_text=output.final_answer,
                tool_calls=case.get("tool_records"),
            )
            risk = eval_res.overall_risk_score or 0.0
            run_latencies.append(output.latency_ms)
            run_tin.append(output.tokens_in)
            run_tout.append(output.tokens_out)
            run_risks.append(risk)
            contra_p = eval_res.grounding.contradiction_prob if eval_res.grounding else 0.0
            run_contra.append(1.0 if (contra_p or 0) > 0.60 else 0.0)

        print(f"  {case['id']}: lat={statistics.mean(run_latencies):.0f}ms  risk={statistics.mean(run_risks):.3f}")

        results_by_strategy[strat_name].append({
            "case_id": case["id"],
            "domain": case["domain"],
            "is_failure_ground_truth": case["is_failure"],
            "n_runs": N_RUNS,
            "avg_latency_ms": round(statistics.mean(run_latencies), 2),
            "median_latency_ms": round(statistics.median(run_latencies), 2),
            "stdev_latency_ms": _stdev(run_latencies),
            "avg_tokens_in": round(statistics.mean(run_tin), 1),
            "avg_tokens_out": round(statistics.mean(run_tout), 1),
            "stdev_tokens_out": _stdev(run_tout),
            "avg_risk_score": round(statistics.mean(run_risks), 3),
            "stdev_risk_score": round(_stdev(run_risks), 3),
            "contradiction_rate": round(statistics.mean(run_contra), 3),
            "raw_latencies_ms": [round(x, 2) for x in run_latencies],
            "raw_risk_scores": [round(x, 3) for x in run_risks],
        })

total_wall_s = time.perf_counter() - overall_start
print(f"\nTotal wall time: {total_wall_s/60:.1f} minutes")

In [ ]:
summary = {}
for s_name, case_results in results_by_strategy.items():
    all_lats = [c["avg_latency_ms"] for c in case_results]
    all_tin = [c["avg_tokens_in"] for c in case_results]
    all_tout = [c["avg_tokens_out"] for c in case_results]
    all_risks = [c["avg_risk_score"] for c in case_results]
    all_contras = [c["contradiction_rate"] for c in case_results]

    summary[s_name.upper()] = {
        "mean_latency_ms": round(statistics.mean(all_lats), 2),
        "median_latency_ms": round(statistics.median(all_lats), 2),
        "stdev_latency_ms": round(float(statistics.stdev(all_lats)), 2) if len(all_lats) > 1 else 0.0,
        "mean_tokens_in": round(statistics.mean(all_tin), 1),
        "mean_tokens_out": round(statistics.mean(all_tout), 1),
        "mean_grounding_risk": round(statistics.mean(all_risks), 3),
        "stdev_grounding_risk": round(float(statistics.stdev(all_risks)), 3) if len(all_risks) > 1 else 0.0,
        "contradiction_rate": round(statistics.mean(all_contras), 3),
    }

out_payload = {
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
    "model": "qwen3-8b-gpu",
    "model_id": adapter.model_id,
    "adapter": type(adapter).__name__,
    "provider": "kaggle_gpu_4bit",
    "real_inference": True,
    "warmup_ms": round(warm.latency_ms, 2),
    "hardware": {
        "platform": "Kaggle",
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
        "quantization": "4-bit (bitsandbytes)",
    },
    "dataset": "v1.0_test",
    "n_cases": len(cases),
    "runs_per_case": N_RUNS,
    "max_tokens_per_call": MAX_TOKENS,
    "total_wall_time_minutes": round(total_wall_s / 60, 1),
    "summary": summary,
    "detailed_results": results_by_strategy,
}

out_path = "/kaggle/working/reasoning_strategy_results.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(out_payload, f, indent=2)

print(f"Saved to {out_path} -- download it from the Output panel.")
print(json.dumps(summary, indent=2))